In [1]:
import gseapy as gp
import pandas as pd
import numpy as np
import time
import os

DATA_DIR    = '/rds/homes/j/jxt554/data'
TABLES_DIR  = '/rds/homes/j/jxt554/tables'
FIGURES_DIR = '/rds/homes/j/jxt554/figures'

print('ORA PATHWAY ENRICHMENT')
print('='*55)

DATABASES = [
    ('Reactome_2022',              'Reactome'),
    ('GO_Biological_Process_2023', 'GO_BP'),
    ('KEGG_2021_Human',            'KEGG'),
    ('MSigDB_Hallmark_2020',       'Hallmark'),
]

def run_ora(gene_list, label, min_genes=5):
    if len(gene_list) < min_genes:
        print(f'  {label}: too few genes '
              f'({len(gene_list)})')
        return {}

    results = {}
    print(f'\nORA: {label} '
          f'({len(gene_list)} proteins)...')

    for db_key, db_short in DATABASES:
        time.sleep(2)
        try:
            enr = gp.enrichr(
                gene_list=gene_list,
                gene_sets=db_key,
                organism='human',
                outdir=None,
                verbose=False)
            res = enr.results.copy()
            res['Term_clean'] = (
                res['Term']
                .str.replace(
                    r'\s+R-HSA-\d+','',
                    regex=True)
                .str.replace(
                    r'\s+\(GO:\d+\)','',
                    regex=True)
                .str.strip())
            sig = res[
                res['Adjusted P-value']<0.05
            ].sort_values('Adjusted P-value')
            print(f'  {db_short}: '
                  f'{len(sig)} pathways')
            if len(sig) > 0:
                top = sig.iloc[0]
                print(f'    Top: '
                      f'{str(top["Term_clean"])[:50]}'
                      f' FDR='
                      f'{top["Adjusted P-value"]:.2e}')
            results[db_short] = sig
            safe  = label.replace(' ','_')
            sig.to_csv(
                f'{TABLES_DIR}/'
                f'ora_{safe}_{db_short}.csv',
                index=False,
                encoding='utf-8')
        except Exception as e:
            print(f'  {db_short}: failed — {e}')

    return results

# ── Load DE results ───────────────────────────────
ora_results = {}

for cohort in ['cd','uc']:
    limma = pd.read_csv(
        f'{TABLES_DIR}/'
        f'{cohort}_limma_M3.csv')

    # C1 markers (up in C1 = logFC < 0)
    c1_up = limma[
        (limma['adj.P.Val'] < 0.05) &
        (limma['logFC'] < 0)
    ]['protein'].tolist()

    # C2 markers (up in C2 = logFC > 0)
    c2_up = limma[
        (limma['adj.P.Val'] < 0.05) &
        (limma['logFC'] > 0)
    ]['protein'].tolist()

    print(f'\n{cohort.upper()}:')
    print(f'  C1 markers: {len(c1_up)}')
    print(f'  C2 markers: {len(c2_up)}')

    ora_c1 = run_ora(
        c1_up, f'{cohort}_C1_up')
    ora_c2 = run_ora(
        c2_up, f'{cohort}_C2_up')

    ora_results[cohort] = {
        'C1': ora_c1,
        'C2': ora_c2,
        'c1_proteins': c1_up,
        'c2_proteins': c2_up}

ORA PATHWAY ENRICHMENT

CD:
  C1 markers: 612
  C2 markers: 3

ORA: cd_C1_up (612 proteins)...
  Reactome: 181 pathways
    Top: Immune System FDR=3.33e-43
  GO_BP: 627 pathways
    Top: Cytokine-Mediated Signaling Pathway FDR=2.72e-27
  KEGG: 109 pathways
    Top: Cytokine-cytokine receptor interaction FDR=9.03e-40
  Hallmark: 32 pathways
    Top: Epithelial Mesenchymal Transition FDR=9.50e-21
  cd_C2_up: too few genes (3)

UC:
  C1 markers: 633
  C2 markers: 14

ORA: uc_C1_up (633 proteins)...
  Reactome: 192 pathways
    Top: Immune System FDR=6.80e-46
  GO_BP: 720 pathways
    Top: Cytokine-Mediated Signaling Pathway FDR=1.77e-28
  KEGG: 101 pathways
    Top: Cytokine-cytokine receptor interaction FDR=5.73e-43
  Hallmark: 31 pathways
    Top: Allograft Rejection FDR=3.36e-20

ORA: uc_C2_up (14 proteins)...
  Reactome: 5 pathways
    Top: Metabolism FDR=7.18e-04
  GO_BP: 39 pathways
    Top: Cellular Response To Catecholamine Stimulus FDR=8.96e-03
  KEGG: 4 pathways
    Top: Nitroge

In [2]:
# ── ORA Summary ───────────────────────────────────
print('\nORA SUMMARY')
print('='*55)
print(f'{"Cohort":<6} {"Contrast":<12} '
      f'{"Reactome":>10} {"GO_BP":>8} '
      f'{"KEGG":>8} {"Hallmark":>10}')
print('-'*56)

for cohort in ['cd','uc']:
    for contrast in ['C1','C2']:
        res = ora_results[cohort][contrast]
        vals = [len(res.get(db,pd.DataFrame()))
                 for db in
                 ['Reactome','GO_BP',
                  'KEGG','Hallmark']]
        print(f'{cohort.upper():<6} '
              f'{contrast+" up":<12} '
              f'{vals[0]:>10} {vals[1]:>8} '
              f'{vals[2]:>8} {vals[3]:>10}')

# Print top Hallmark pathways
for cohort in ['cd','uc']:
    for contrast in ['C1','C2']:
        hm = ora_results[cohort][
            contrast].get(
            'Hallmark', pd.DataFrame())
        if len(hm) > 0:
            print(f'\nTop Hallmark — '
                  f'{cohort.upper()} '
                  f'{contrast} up:')
            for _, r in hm.head(8).iterrows():
                print(f'  '
                      f'{str(r["Term_clean"])[:50]}'
                      f' FDR='
                      f'{r["Adjusted P-value"]:.2e}')

print('\nORA COMPLETE')
print('Next: PPI network analysis')


ORA SUMMARY
Cohort Contrast       Reactome    GO_BP     KEGG   Hallmark
--------------------------------------------------------
CD     C1 up               181      627      109         32
CD     C2 up                 0        0        0          0
UC     C1 up               192      720      101         31
UC     C2 up                 5       39        4          8

Top Hallmark — CD C1 up:
  Epithelial Mesenchymal Transition FDR=9.50e-21
  Allograft Rejection FDR=3.44e-19
  Apoptosis FDR=3.10e-15
  Interferon Gamma Response FDR=3.41e-15
  Complement FDR=3.41e-15
  IL-6/JAK/STAT3 Signaling FDR=8.46e-15
  Inflammatory Response FDR=1.66e-14
  TNF-alpha Signaling via NF-kB FDR=5.99e-13

Top Hallmark — UC C1 up:
  Allograft Rejection FDR=3.36e-20
  Epithelial Mesenchymal Transition FDR=1.40e-19
  Inflammatory Response FDR=4.44e-17
  Interferon Gamma Response FDR=2.43e-16
  Apoptosis FDR=2.63e-13
  Complement FDR=1.78e-12
  IL-6/JAK/STAT3 Signaling FDR=1.78e-12
  TNF-alpha Signaling via N